# Logistic Regression: Binary and Multiclass Classification

## Overview
Despite its name, logistic regression is a **classification** algorithm. It models the probability that an instance belongs to a particular class using the logistic (sigmoid) function.

## Learning Objectives
- Understand the sigmoid function and its properties
- Derive the logistic regression cost function (log loss)
- Implement logistic regression from scratch
- Extend to multiclass classification (one-vs-rest, multinomial)
- Evaluate classification models using appropriate metrics
- Apply regularization to prevent overfitting

## Mathematical Foundation

### Sigmoid Function
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

### Hypothesis
$$h_\theta(x) = \sigma(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

### Decision Boundary
- Predict class 1 if $h_\theta(x) \geq 0.5$ (i.e., $\theta^T x \geq 0$)
- Predict class 0 if $h_\theta(x) < 0.5$ (i.e., $\theta^T x < 0$)

### Cost Function (Log Loss / Binary Cross-Entropy)
$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}[y^{(i)}\log(h_\theta(x^{(i)})) + (1-y^{(i)})\log(1-h_\theta(x^{(i)}))]$$

### Gradient
$$\frac{\partial J}{\partial \theta_j} = \frac{1}{m}\sum_{i=1}^{m}(h_\theta(x^{(i)}) - y^{(i)})x_j^{(i)}$$

Note: Same form as linear regression, but $h_\theta$ is different!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, precision_recall_curve)
from sklearn.datasets import make_classification, load_iris, load_breast_cancer

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Understanding the Sigmoid Function

In [ ]:
def sigmoid(z):
    """Sigmoid (logistic) function"""
    return 1 / (1 + np.exp(-z))

# Visualize sigmoid function
z = np.linspace(-10, 10, 200)
sigma_z = sigmoid(z)

plt.figure(figsize=(12, 5))

# Sigmoid function
plt.subplot(1, 2, 1)
plt.plot(z, sigma_z, 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Decision threshold')
plt.axvline(x=0, color='g', linestyle='--', alpha=0.5, label='z = 0')
plt.xlabel('z = θᵀx', fontsize=12)
plt.ylabel('σ(z)', fontsize=12)
plt.title('Sigmoid Function', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()

# Derivative
plt.subplot(1, 2, 2)
derivative = sigma_z * (1 - sigma_z)
plt.plot(z, derivative, 'r-', linewidth=2)
plt.xlabel('z', fontsize=12)
plt.ylabel("σ'(z) = σ(z)(1-σ(z))", fontsize=12)
plt.title('Sigmoid Derivative', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Properties of Sigmoid:")
print(f"  σ(-∞) = 0")
print(f"  σ(0) = {sigmoid(0):.1f}")
print(f"  σ(+∞) = 1")
print(f"  Output range: (0, 1) - interpreted as probability!")
print(f"  Derivative: σ'(z) = σ(z)(1 - σ(z))")

## 2. Binary Classification from Scratch

In [ ]:
class LogisticRegressionScratch:
    """Logistic Regression from scratch using gradient descent"""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.theta = None
        self.cost_history = []
    
    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # Clip to avoid overflow
    
    def _compute_cost(self, X, y, theta):
        m = len(y)
        h = self._sigmoid(X @ theta)
        epsilon = 1e-15  # To avoid log(0)
        cost = -(1/m) * np.sum(y * np.log(h + epsilon) + (1-y) * np.log(1-h + epsilon))
        return cost
    
    def fit(self, X, y):
        m, n = X.shape
        
        # Add bias term
        X_b = np.c_[np.ones((m, 1)), X]
        
        # Initialize parameters
        self.theta = np.zeros((n + 1, 1))
        
        # Gradient descent
        for i in range(self.n_iterations):
            # Predictions
            h = self._sigmoid(X_b @ self.theta)
            
            # Compute cost
            cost = self._compute_cost(X_b, y, self.theta)
            self.cost_history.append(cost)
            
            # Compute gradients
            gradients = (1/m) * X_b.T @ (h - y)
            
            # Update parameters
            self.theta -= self.learning_rate * gradients
        
        return self
    
    def predict_proba(self, X):
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return self._sigmoid(X_b @ self.theta)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("✓ Logistic Regression class implemented from scratch")

## 3. Testing on Synthetic Data

In [ ]:
# Generate binary classification dataset
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1,
                          random_state=42)
y = y.reshape(-1, 1)

# Visualize data
plt.figure(figsize=(10, 6))
plt.scatter(X[y.ravel()==0, 0], X[y.ravel()==0, 1], 
           c='blue', marker='o', s=50, alpha=0.7, edgecolors='k', label='Class 0')
plt.scatter(X[y.ravel()==1, 0], X[y.ravel()==1, 1], 
           c='red', marker='s', s=50, alpha=0.7, edgecolors='k', label='Class 1')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Binary Classification Dataset', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train from-scratch model
model_scratch = LogisticRegressionScratch(learning_rate=0.1, n_iterations=1000)
model_scratch.fit(X_train_scaled, y_train)

# Predictions
y_pred = model_scratch.predict(X_test_scaled)
y_proba = model_scratch.predict_proba(X_test_scaled)

print("From-Scratch Logistic Regression Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score: {f1_score(y_test, y_pred):.4f}")

In [ ]:
# Visualize cost convergence
plt.figure(figsize=(10, 6))
plt.plot(model_scratch.cost_history, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Cost (Log Loss)', fontsize=12)
plt.title('Cost Function Convergence', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial cost: {model_scratch.cost_history[0]:.4f}")
print(f"Final cost: {model_scratch.cost_history[-1]:.4f}")

## 4. Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(X, y, model, title="Decision Boundary"):
    """Plot the decision boundary for a 2D dataset"""
    h = 0.02  # Step size in mesh
    
    # Create mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    plt.scatter(X[y.ravel()==0, 0], X[y.ravel()==0, 1], 
               c='blue', marker='o', s=50, edgecolors='k', label='Class 0')
    plt.scatter(X[y.ravel()==1, 0], X[y.ravel()==1, 1], 
               c='red', marker='s', s=50, edgecolors='k', label='Class 1')
    plt.xlabel('Feature 1', fontsize=12)
    plt.ylabel('Feature 2', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.show()

# Plot decision boundary
plot_decision_boundary(X_train_scaled, y_train, model_scratch, 
                      "Logistic Regression: Decision Boundary")

## 5. Using Scikit-learn

In [ ]:
# Train sklearn model
model_sklearn = LogisticRegression(random_state=42)
model_sklearn.fit(X_train_scaled, y_train.ravel())

# Predictions
y_pred_sklearn = model_sklearn.predict(X_test_scaled)
y_proba_sklearn = model_sklearn.predict_proba(X_test_scaled)[:, 1]

print("Scikit-learn Logistic Regression Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_sklearn):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_sklearn):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred_sklearn):.4f}")
print(f"  F1-Score: {f1_score(y_test, y_pred_sklearn):.4f}")
print(f"\nModel parameters:")
print(f"  Intercept: {model_sklearn.intercept_[0]:.4f}")
print(f"  Coefficients: {model_sklearn.coef_[0]}")

## 6. Confusion Matrix and Classification Report

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_sklearn)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
           xticklabels=['Class 0', 'Class 1'],
           yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred_sklearn, 
                          target_names=['Class 0', 'Class 1']))

## 7. ROC Curve and AUC

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba_sklearn)
roc_auc = roc_auc_score(y_test, y_proba_sklearn)

# Precision-Recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba_sklearn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall, precision, 'g-', linewidth=2)
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUC-ROC Score: {roc_auc:.4f}")
print(f"\nInterpretation: AUC closer to 1.0 indicates better classification")

## 8. Multiclass Classification

In [ ]:
# Load Iris dataset (3 classes)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Iris Dataset:")
print(f"  Samples: {X_iris.shape[0]}")
print(f"  Features: {X_iris.shape[1]}")
print(f"  Classes: {len(np.unique(y_iris))} - {iris.target_names}")

In [ ]:
# One-vs-Rest (OvR) strategy
model_ovr = LogisticRegression(multi_class='ovr', random_state=42)
model_ovr.fit(X_train_scaled, y_train)
y_pred_ovr = model_ovr.predict(X_test_scaled)

# Multinomial strategy
model_multi = LogisticRegression(multi_class='multinomial', random_state=42)
model_multi.fit(X_train_scaled, y_train)
y_pred_multi = model_multi.predict(X_test_scaled)

print("Multiclass Classification Results:")
print(f"\nOne-vs-Rest Strategy:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_ovr):.4f}")
print(f"\nMultinomial Strategy:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_multi):.4f}")

In [ ]:
# Confusion matrix for multiclass
cm_multi = confusion_matrix(y_test, y_pred_multi)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues', cbar=False,
           xticklabels=iris.target_names,
           yticklabels=iris.target_names)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Multiclass Confusion Matrix', fontsize=14)
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred_multi, target_names=iris.target_names))

## 9. Regularization in Logistic Regression

In [ ]:
# Compare different regularization strengths
C_values = [0.001, 0.01, 0.1, 1, 10, 100]  # C = 1/λ (inverse regularization)
results_reg = []

for C in C_values:
    model = LogisticRegression(C=C, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    results_reg.append({
        'C': C,
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred)
    })

df_reg = pd.DataFrame(results_reg)
print("Regularization Strength Comparison:")
print(df_reg.to_string(index=False))
print(f"\nNote: Smaller C = stronger regularization")

In [ ]:
# Visualize regularization effect
plt.figure(figsize=(10, 6))
plt.plot(df_reg['C'], df_reg['Train Accuracy'], 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(df_reg['C'], df_reg['Test Accuracy'], 'ro-', label='Test Accuracy', linewidth=2)
plt.xscale('log')
plt.xlabel('C (Inverse Regularization Strength)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of Regularization on Model Performance', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 10. Real-World Example: Breast Cancer Detection

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Breast Cancer Dataset:")
print(f"  Samples: {X_cancer.shape[0]}")
print(f"  Features: {X_cancer.shape[1]}")
print(f"  Classes: Malignant (0), Benign (1)")
print(f"  Class distribution: {np.bincount(y_cancer)}")

In [ ]:
# Train model
model_cancer = LogisticRegression(random_state=42, max_iter=10000)
model_cancer.fit(X_train_scaled, y_train)

# Predictions
y_pred = model_cancer.predict(X_test_scaled)
y_proba = model_cancer.predict_proba(X_test_scaled)[:, 1]

print("Breast Cancer Detection Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred):.4f}")
print(f"  F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"  AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

print("\nFor medical diagnosis:")
print("  High recall is critical (don't miss cancer cases)")
print("  High precision is also important (avoid false alarms)")

## Key Takeaways

### Advantages of Logistic Regression
✅ Simple and interpretable
✅ Outputs probabilities (useful for uncertainty quantification)
✅ Fast training and prediction
✅ Works well with linearly separable data
✅ Naturally extends to multiclass (OvR, multinomial)
✅ Built-in regularization (L1, L2)

### Limitations
❌ Assumes linear decision boundary
❌ Can underfit complex non-linear patterns
❌ Sensitive to outliers
❌ Requires feature scaling

### Important Concepts
- **Decision boundary**: Linear in feature space
- **Log loss**: Appropriate cost function for classification
- **Probability calibration**: Outputs are well-calibrated probabilities
- **Regularization**: Prevents overfitting (especially with many features)
- **Multiclass**: OvR creates N binary classifiers, multinomial is more elegant

## Next Steps
- **Decision Trees**: Handle non-linear boundaries
- **Random Forests**: Ensemble of trees
- **SVM**: Alternative margin-based classifier
- **Neural Networks**: Deep learning approaches